In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [12]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd

# creates the client
woc = WocMapsRemote(base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    "arnaudon/netSALT",
    "PythonOT/POT",
    "blegat/SwitchOnSafety.jl",
    "connectomicslab/connectomemapper3",
    "BBN-Q/Auspex",
    "vsamy/Copra",
    "havakv/pycox",
    "seung-lab/igneous",
    "salan668/FAE",
    "matsengrp/transport",
]
list_df_commits = []
for prj in projects:
    # Convert GitHub repository names to WoC V2412 project names
    prj = prj.lower().replace("/", "_", 2)
    commits = woc.get_values("p2c", prj)
    df = pd.DataFrame(commits, columns=["sha1"])
    df["project"] = prj
    list_df_commits.append(df)
df = pd.concat(list_df_commits)
# perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv("df_commits.csv")
df.head(1)

,sha1,project
0,005227b76ce4b178c1e5cbf5d4ff2a92fed36456,arnaudon_netsalt


In [ ]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

In [1]:
import pandas as pd
netid = 'pshu'

# summary csv
data = pd.read_csv('df_commit_data.csv')
summary = data[['project', 'commit', 'author', 'author_time', 'message']]
summary.columns = ['project_wocid', 'commit_sha1', 'author', 'time', 'message']
summary.to_csv(netid + '_project_summary.csv', index=False, sep=';')

chk = pd.read_csv(netid + '_project_summary.csv', sep=';')
print(len(chk), chk.project_wocid.nunique(), chk.commit_sha1.is_unique)

17052 10 True


In [2]:
# woc stats
g = summary.groupby('project_wocid')
woc_stats = pd.DataFrame({
    'ncommits': g.commit_sha1.nunique(),
    'nauthors': g.author.nunique(),
    'min_time': g.time.min(),
    'max_time': g.time.max(),
}).reset_index()
woc_stats['from'] = pd.to_datetime(woc_stats.min_time, unit='s').dt.strftime('%Y-%m-%d')
woc_stats['to'] = pd.to_datetime(woc_stats.max_time, unit='s').dt.strftime('%Y-%m-%d')
woc_stats

,project_wocid,ncommits,nauthors,min_time,max_time,from,to
0,arnaudon_netsalt,425,15,1550502528,1733404891,2019-02-18,2024-12-05
1,bbn-q_auspex,3636,75,1441982889,1748609937,2015-09-11,2025-05-30
2,blegat_switchonsafety.jl,545,9,1474792981,1724336641,2016-09-25,2024-08-22
3,connectomicslab_connectomemapper3,3789,30,1348156133,1774022355,2012-09-20,2026-03-20
4,havakv_pycox,414,38,1518407518,1761834963,2018-02-12,2025-10-30
5,matsengrp_transport,307,10,1557872487,1724504919,2019-05-14,2024-08-24
6,pythonot_pot,5212,190,1476974534,1775826870,2016-10-20,2026-04-10
7,salan668_fae,510,30,1528680759,1756716898,2018-06-11,2025-09-01
8,seung-lab_igneous,1897,37,1513298507,1761161263,2017-12-15,2025-10-22
9,vsamy_copra,317,14,1476116832,1753237340,2016-10-10,2025-07-23


In [4]:
import requests, time

repos = {
    'arnaudon_netsalt': 'arnaudon/netSALT',
    'pythonot_pot': 'PythonOT/POT',
    'blegat_switchonsafety.jl': 'blegat/SwitchOnSafety.jl',
    'connectomicslab_connectomemapper3': 'connectomicslab/connectomemapper3',
    'bbn-q_auspex': 'BBN-Q/Auspex',
    'vsamy_copra': 'vsamy/Copra',
    'havakv_pycox': 'havakv/pycox',
    'seung-lab_igneous': 'seung-lab/igneous',
    'salan668_fae': 'salan668/FAE',
    'matsengrp_transport': 'matsengrp/transport',
}

# gh api
rows = []
for prj in repos:
    url = 'https://api.github.com/repos/' + repos[prj]
    r = requests.get(url).json()
    cm = requests.get(url + '/commits?per_page=1').json()
    rows.append([prj, r['stargazers_count'], r['forks_count'], cm[0]['commit']['committer']['date'][:10]])
    time.sleep(1)

gh_stats = pd.DataFrame(rows, columns=['project', 'nstars', 'nforks', 'lastGHCommitDate'])
gh_stats

,project,nstars,nforks,lastGHCommitDate
0,arnaudon_netsalt,9,4,2026-08-14
1,pythonot_pot,2848,569,2026-10-05
2,blegat_switchonsafety.jl,24,2,2024-08-14
3,connectomicslab_connectomemapper3,87,31,2025-02-04
4,bbn-q_auspex,34,15,2026-06-18
5,vsamy_copra,43,10,2026-07-04
6,havakv_pycox,996,203,2024-09-04
7,seung-lab_igneous,69,24,2026-09-30
8,salan668_fae,172,50,2026-09-26
9,matsengrp_transport,4,0,2024-08-24


In [ ]:
# part 1 csv
stats = woc_stats.rename(columns={'project_wocid': 'project'}).merge(gh_stats, on='project')
stats = stats[['project', 'ncommits', 'nauthors', 'from', 'to', 'nstars', 'nforks', 'lastGHCommitDate']]
stats.to_csv(netid + '_project_stats.csv', index=False, sep=';')
stats

## Part 1 summary

Commit counts, author counts, and first/last commit dates come from World of Code. Stars, forks, and last commit date come from the GitHub API.

| project | commits | authors | from | to | stars | forks | last GH commit |
|---|---|---|---|---|---|---|---|
| arnaudon_netsalt | 425 | 15 | 2019-02-18 | 2024-12-05 | 9 | 4 | 2026-08-14 |
| bbn-q_auspex | 3636 | 75 | 2015-09-11 | 2025-05-30 | 34 | 15 | 2026-06-18 |
| blegat_switchonsafety.jl | 545 | 9 | 2016-09-25 | 2024-08-22 | 24 | 2 | 2024-08-14 |
| connectomicslab_connectomemapper3 | 3789 | 30 | 2012-09-20 | 2026-03-20 | 87 | 31 | 2025-02-04 |
| havakv_pycox | 414 | 38 | 2018-02-12 | 2025-10-30 | 996 | 203 | 2024-09-04 |
| matsengrp_transport | 307 | 10 | 2019-05-14 | 2024-08-24 | 4 | 0 | 2024-08-24 |
| pythonot_pot | 5212 | 190 | 2016-10-20 | 2026-04-10 | 2848 | 569 | 2026-10-05 |
| salan668_fae | 510 | 30 | 2018-06-11 | 2025-09-01 | 172 | 50 | 2026-09-26 |
| seung-lab_igneous | 1897 | 37 | 2017-12-15 | 2025-10-22 | 69 | 24 | 2026-09-30 |
| vsamy_copra | 317 | 14 | 2016-10-10 | 2025-07-23 | 43 | 10 | 2026-07-04 |

WoC listed 17,093 commits for these ten projects, but 41 keys were not found in commit.tch while fetching the commit contents, so 17,052 commits are in the summary csv (vsamy_copra 29, bbn-q_auspex 4, seung-lab_igneous 4, blegat_switchonsafety.jl 2, salan668_fae 1).

For connectomicslab_connectomemapper3, havakv_pycox, and blegat_switchonsafety.jl the latest WoC commit is later than the last commit GitHub reports for the default branch. Possibly commits from other branches or forks that WoC groups into the same project. Gap and time series results below use the WoC data.

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github